In [1]:
import pandas as pd
import os
import numpy as np
from scipy.stats import spearmanr, ttest_ind
import itertools

In [2]:
def min_max_norm(data):
    data_min = np.min(data)
    data_max = np.max(data)
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

In [3]:
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
attn_dir = "results/attention_maps"
data_dir = "data"
task_type = "classification"
info_df = pd.read_csv(os.path.join(root_dir, data_dir, "input_info_VRC01_IC80.csv"))
info_df.sort_values(by=["Subtype_Label", "CRF", "Subtype", "Seq_name"], inplace=True)
aligned_sequences = np.array([list(seq) for seq in info_df['Sequence']])
resno_df = pd.read_csv(os.path.join(root_dir, data_dir, "selected_residues_IC80.csv"))
resno_array = resno_df["ResLabel"].values

In [4]:
N_SEQ, L_SEQ = len(info_df), aligned_sequences.shape[1]
N_MODEL = len(selected_models)

all_cls_attn_array =  np.zeros([N_MODEL, N_SEQ, L_SEQ])
all_attn_array =  np.zeros([N_MODEL, N_SEQ, L_SEQ, L_SEQ])
variant_idx = []
for n, model_name in enumerate(selected_models):
    for i, no in enumerate(info_df['Seq_no']):
        attn = np.load(os.path.join(root_dir, attn_dir, task_type, "full", model_name, str(no)+"_attentions.npy"))
        cls_attn = attn[0, 1:]
        all_cls_attn_array[n,i] = min_max_norm(cls_attn)
        attn_arr = attn[1:, 1:]
        all_attn_array[n,i] = min_max_norm(attn_arr)

In [6]:
arr_dict = {
    "[CLS]-to-Residue Attention" : all_cls_attn_array,
    "Residue-Residue Attention": all_attn_array
}

for name, arr in arr_dict.items():
    print(name)
    reshaped_array = arr.reshape(N_MODEL, -1)
    corr, pvalue = spearmanr(reshaped_array, axis=1)
    print("Correlation Matrix:\n", np.round(corr, 2))
    print("Mean off-diag corr:", np.mean(corr[np.triu_indices_from(corr, k=1)]).round(2))
    print("-------")

[CLS]-to-Residue Attention
Correlation Matrix:
 [[1.   0.68 0.65 0.63 0.62]
 [0.68 1.   0.69 0.65 0.61]
 [0.65 0.69 1.   0.62 0.68]
 [0.63 0.65 0.62 1.   0.71]
 [0.62 0.61 0.68 0.71 1.  ]]
Mean off-diag corr: 0.65
-------
Residue-Residue Attention
Correlation Matrix:
 [[1.   0.68 0.65 0.63 0.62]
 [0.68 1.   0.69 0.65 0.61]
 [0.65 0.69 1.   0.62 0.68]
 [0.63 0.65 0.62 1.   0.71]
 [0.62 0.61 0.68 0.71 1.  ]]
Mean off-diag corr: 0.65
-------


In [6]:
vrc01_contact_residues = [97, 105, 123, 192, 193, 198, 199, 276, 278, 279, 280, 281, 282, 283, 362, 365, 366, 367, 368, 371, 427, 428, 429, 430, 455, 456, 457, 458, 459, 460, 461, 463, 465, 466, 467, 469, 472, 473, 474, 476]
vrc01_contact_residues = [str(_) for _ in vrc01_contact_residues]
mask_vrc01_contact_residues = np.isin(resno_array, vrc01_contact_residues)

epitopes = resno_df["Epitope"].unique()
epitope_dfs = resno_df.groupby(by="Epitope")

name = "[CLS]-to-Residue Attention"
arr = arr_dict[name].copy()
mean_arr = np.mean(arr, axis=0)

all_samples = {}

# Get VRC01 and Non-VRC01 Samples
vrc01_samples = (np.sum(mean_arr[:,mask_vrc01_contact_residues], axis=(-1)) / mask_vrc01_contact_residues.sum()).flatten()
all_samples['VRC01 Contact'] = vrc01_samples

non_vrc01_samples = (np.sum(mean_arr[:,~mask_vrc01_contact_residues], axis=(-1)) / (~mask_vrc01_contact_residues).sum()).flatten()
all_samples['Non-VRC01 Contact'] = non_vrc01_samples

# Get Epitope Samples
for epitope in epitopes:
    epitope_reslabel = epitope_dfs.get_group(epitope)["ResLabel"].unique()
    mask_epitope_respos = np.isin(resno_array, epitope_reslabel)
    if mask_epitope_respos.sum() == 0:
        continue
    epitope_samples = (np.sum(mean_arr[:,mask_epitope_respos], axis=(-1)) / mask_epitope_respos.sum()).flatten()
    group_name = f'Epitope: {epitope.replace("_","-")}'
    all_samples[group_name] = epitope_samples
    
stats_dict = {}
for group_name, data in all_samples.items():
    stats_dict[group_name] = {
        'mean': np.mean(data),
        'sd': np.std(data)
    }
stats_df = pd.DataFrame.from_dict(stats_dict, orient='index')

print("\n--- Statistics Table ---")
print(stats_df)

group_names = list(all_samples.keys())
num_groups = len(group_names)

p_value_matrix = pd.DataFrame(np.nan, index=group_names, columns=group_names)
annot_matrix = pd.DataFrame("", index=group_names, columns=group_names)

# Calculate all pairwise p-values
for group_1, group_2 in itertools.combinations(group_names, 2):
    samples_1 = all_samples[group_1]
    samples_2 = all_samples[group_2]
    
    _, p_value = ttest_ind(samples_1, samples_2, equal_var=False)
    
    p_value_matrix.loc[group_1, group_2] = p_value
    p_value_matrix.loc[group_2, group_1] = p_value

print("\n--- P-value ---")
print(p_value_matrix.to_string())


--- Statistics Table ---
                               mean        sd
Epitope: CD4-binding-site  0.274229  0.043505
Epitope: V2-apex           0.273189  0.058476
Epitope: V3-glycan         0.232568  0.047657
Epitope: interface         0.280887  0.036143
Non-VRC01 Contact          0.260281  0.032692
VRC01 Contact              0.309397  0.047031

--- P-value ---
                           VRC01 Contact  Non-VRC01 Contact  Epitope: interface  Epitope: CD4-binding-site  Epitope: V2-apex  Epitope: V3-glycan
VRC01 Contact                        NaN      7.052991e-121        5.725660e-44               4.159926e-56      2.543722e-44       2.632717e-197
Non-VRC01 Contact          7.052991e-121                NaN        6.512268e-35               3.721449e-14      1.149673e-08        1.215440e-43
Epitope: interface          5.725660e-44       6.512268e-35                 NaN               4.634370e-04      8.681212e-04       4.864284e-110
Epitope: CD4-binding-site   4.159926e-56       3.721449